# 04 — build_agent.py prototype

Goal: stop hand-duplicating `wheat_loop_agent` between
`src/kaggriculture_bot/strategies/baseline.py` and `agent/main.py`.
Prototype the generator logic here first; once each piece works, it
graduates into `scripts/build_agent.py` (per `structuring-ml-projects`'s
notebook-before-src rule).

Design reference: `docs/superpowers/specs/2026-08-12-build-agent-script-design.md`.

Plan: read the source of `wheat_loop_agent` from `baseline.py`, pull out
just the function's code, rename it to `agent`, and write it into a
string that becomes the new `agent/main.py` content.

In [114]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd().parent / "src"))
source = (Path.cwd().parent / "src" / "kaggriculture_bot" / "strategies" / "baseline.py").read_text()

In [115]:
import ast

tree = ast.parse(source)

for b in tree.body:
    if (isinstance(b, ast.FunctionDef)) and b.name == "wheat_loop_agent":
        idx_0 = b.lineno -1
        idx_end = b.end_lineno
        list_source = source.splitlines(keepends=True)[idx_0:idx_end]
        print("".join(list_source))



def wheat_loop_agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = ["PASS"]
    else:
        farmer = ["PASS"]

    return {"farmer": farmer, "hands": [], "market": market}



In [116]:
import ast

def function_source(source, func_name):

    tree = ast.parse(source)

    for b in tree.body:
        if (isinstance(b, ast.FunctionDef)) and b.name == func_name:
            idx_0 = b.lineno -1
            idx_end = b.end_lineno
            list_source = source.splitlines(keepends=True)[idx_0:idx_end]

    return "".join(list_source)

agent_source = function_source(source, "wheat_loop_agent").replace("def wheat_loop_agent", "def agent")
print(agent_source)

def agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = ["PASS"]
    else:
        farmer = ["PASS"]

    return {"farmer": farmer, "hands": [], "market": market}



In [117]:
header = '''"""Submission entrypoint. Must define `agent(obs)` -> action dict.

GENERATED FILE -- do not edit by hand. Edit the strategy in
src/kaggriculture_bot/strategies/ and re-run the generator (this
notebook / scripts/build_agent.py once it graduates).

This is the "production" pipeline: only strategies that graduated from
notebooks/ + evaluate.py (see RESOURCES.md, PROGRESS.md) get wired in here.
Packaged and submitted via `kaggle competitions submit kaggriculture -f ...`
(see AGENTS.md). No network/file I/O here at inference time — see
docs/RULES_SUMMARY.md (No Ingress or Egress).
"""'''

In [118]:
main_content = header + "\n" + "\n" + "\n" + agent_source
print(main_content)

"""Submission entrypoint. Must define `agent(obs)` -> action dict.

GENERATED FILE -- do not edit by hand. Edit the strategy in
src/kaggriculture_bot/strategies/ and re-run the generator (this
notebook / scripts/build_agent.py once it graduates).

This is the "production" pipeline: only strategies that graduated from
notebooks/ + evaluate.py (see RESOURCES.md, PROGRESS.md) get wired in here.
Packaged and submitted via `kaggle competitions submit kaggriculture -f ...`
(see AGENTS.md). No network/file I/O here at inference time — see
docs/RULES_SUMMARY.md (No Ingress or Egress).
"""


def agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        m

In [119]:
(Path.cwd().parent / "agent" / "main.py").write_text(main_content)

1500

In [120]:
string = "kaggriculture_bot.strategies.baseline:wheat_loop_agent"
module_name, func_name = string.split(":")

In [121]:
module_name_path = module_name.replace(".","/") + ".py"
module_name_path

'kaggriculture_bot/strategies/baseline.py'

In [122]:
module_source = (Path.cwd().parent / "src" / module_name_path).read_text()
module_source

'"""Baseline heuristic agent(s).\n\nwheat_loop_agent follows the "Example — Wheat Loop" from AGENTS.md\n(competition kit, root of this repo): buy wheat seed if out of stock,\nplant on an empty tile, harvest as soon as the crop is ready (checked\nbefore watering, matching AGENTS.md\'s own priority order), otherwise\nwater. Sells wheat one unit per turn rather than dumping the whole shed\nat once — validated as the better choice in\nnotebooks/03_baseline_wheat_loop.ipynb (see 2026-08-10 entry in\nPROGRESS.md: selling the full shed per turn showed no consistent\nimprovement over n=10/n=20 runs vs "starter").\n\nValidated in notebooks/03_baseline_wheat_loop.ipynb via evaluate.py:\nwin_rate=1.0 vs both "random" and "starter" built-in agents.\n"""\n\n\ndef wheat_loop_agent(obs):\n    market = []\n    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)\n    if obs["step"] == 0 or wheat_seeds <= 0:\n        market.append(["BUY_SEED", "WHEAT", 1])\n\n    farmer_pos = obs["farms"][obs["player"

In [123]:
func_source = function_source(module_source, func_name).replace(f"def {func_name}(", "def agent(")
print(func_source)

def agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = ["PASS"]
    else:
        farmer = ["PASS"]

    return {"farmer": farmer, "hands": [], "market": market}



In [124]:
def generate_files(strategy_path):
    module_name, func_name = strategy_path.split(":")
    module_name_path = module_name.replace(".","/") + ".py"
    module_source = (Path.cwd().parent / "src" / module_name_path).read_text()
    func_source = function_source(module_source, func_name).replace(f"def {func_name}(", "def agent(")
    return {"main.py":header + "\n\n\n" + func_source}

generate_files("kaggriculture_bot.strategies.baseline:wheat_loop_agent")["main.py"] == main_content

True

In [125]:
string_prueba = "from kaggriculture_bot.config import CROPS"

tree = ast.parse(string_prueba)
type(tree.body[0])
print(type(tree.body[0]))
tree.body[0].module
tree.body[0].module.startswith("kaggriculture_bot.")

<class 'ast.ImportFrom'>


True

In [126]:
def internal_imports(source):
    tree = ast.parse(source)
    result = []
    for b in tree.body:
            if (isinstance(b, ast.ImportFrom)) and b.module.startswith("kaggriculture_bot."):
                  result.append(b)
    return result

internal_imports(string_prueba)

In [127]:
config = (tree.body[0].module).split(".")[-1]
string_prueba.replace(f"from {tree.body[0].module} import",f"from {config} import")

'from config import CROPS'

In [128]:
def rewrite_internal_imports(source):
    nodes = internal_imports(source)
    for node in nodes:
        flat_name = node.module.split(".")[-1]
        source = source.replace(f"from {node.module} import", f"from {flat_name} import")
    return source
rewrite_internal_imports(string_prueba)

'from config import CROPS'

In [129]:
def top_level_imports(source):
    tree = ast.parse(source)
    result = []
    for b in tree.body:
        if isinstance(b, (ast.Import, ast.ImportFrom)):
            result.append(b)
    return result

top_level_imports(string_prueba)

In [130]:
def top_level_import_block(source):
    rewritten = rewrite_internal_imports(source)
    nodes = top_level_imports(source)
    result = []
    for node in nodes:
        idx_0 = node.lineno -1
        idx_end = node.end_lineno
        list_source = rewritten.splitlines(keepends=True)[idx_0:idx_end]
        result.extend(list_source)
    return "".join(result)
top_level_import_block(string_prueba)

'from config import CROPS'

In [131]:
def collect_internal_modules(module_name, collected):
    if module_name in collected:
        return
    else:
        module_name_path = module_name.replace(".","/") + ".py"
        module_source = (Path.cwd().parent / "src" / module_name_path).read_text()
        collected[module_name] = rewrite_internal_imports(module_source)
        for node in internal_imports(module_source):
            collect_internal_modules(node.module, collected)

collected = {}
collect_internal_modules("kaggriculture_bot.config", collected)
print(collected)

{'kaggriculture_bot.config': '"""Single source of truth for game constants.\n\nRe-exported directly from the installed `kaggle_environments` package\n(confirmed present in kaggle-environments\' kaggriculture module — no need to\nhand-transcribe the README tables, which can drift from the actual engine).\nStrategy code should import constants from here, not from\n`kaggle_environments` directly, so there is one place to update if the\nimport path ever changes.\n"""\n\nfrom kaggle_environments.envs.kaggriculture.kaggriculture import (\n    ANIMALS,\n    CROPS,\n    FARM_HAND_COST_MULT,\n    LAND_PRICES,\n    MARKET_I0,\n    MARKET_PARAMS,\n    PRICE_FLOOR,\n    PRODUCTS,\n    SHOPS,\n    TOWN_CENTER_DEMAND_SCHEDULE,\n)\n\nRANDOM_STATE = 0\n'}


## Wire imports into `generate_files`

Last piece: `generate_files` needs to (1) include the strategy module's
top-level imports (flattened) before the function, and (2) copy any
internal `kaggriculture_bot.*` dependency as its own flat file in the
output dict, using `collect_internal_modules`.

`flat_name` turns a dotted module name into the flat filename it gets in
the output (`"kaggriculture_bot.config"` -> `"config.py"`) — same
`.split(".")[-1]` pattern used inside `collect_internal_modules`, just
pulled out on its own since we need it here too.

In [132]:
def flat_name(module_name):
    return module_name.split(".")[-1] + ".py"

flat_name("kaggriculture_bot.config")

'config.py'

In [133]:
def generate_files(strategy_path):
    module_name, func_name = strategy_path.split(":")
    module_name_path = module_name.replace(".", "/") + ".py"
    module_source = (Path.cwd().parent / "src" / module_name_path).read_text()
    func_source = function_source(module_source, func_name).replace(f"def {func_name}(", "def agent(")

    collected = {}
    for node in internal_imports(module_source):
        collect_internal_modules(node.module, collected)

    import_block = top_level_import_block(module_source)
    main_body = header + "\n\n\n"
    if import_block:
        main_body += import_block + "\n\n"
    main_body += func_source

    files = {"main.py": main_body}
    for dep_name, dep_source in collected.items():
        files[flat_name(dep_name)] = dep_source
    return files

# Regression check: baseline.py has no internal imports today, so this
# should behave exactly like the pre-import-handling version and still
# match main_content.
generate_files("kaggriculture_bot.strategies.baseline:wheat_loop_agent")["main.py"] == main_content

True